In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [2]:
from pathlib import Path

# ==============================================================================
# ⚙️ CẤU HÌNH ĐÁNH GIÁ (CHỈ CẦN CHỌN TẠI ĐÂY)
# ==============================================================================
# Lựa chọn ngôn ngữ test:
#   EVAL_LANGS = ["vi"]          -> Chỉ test Tiếng Việt
#   EVAL_LANGS = ["en"]          -> Chỉ test Tiếng Anh
#   EVAL_LANGS = ["vi", "en"]    -> Tự động test cả 2 và in bảng so sánh đối chiếu
EVAL_LANGS = ["vi", "en"]

EXPERIMENT = "e4"

MODEL_ID = "unsloth/Qwen3.5-2B"


LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_RUN_DIR = Path("/content/run")



LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

HF_NAME = "ThinhDao"

model_name_only = (f"{MODEL_ID.rsplit('/', 1)[-1]}_{EXPERIMENT.upper()}")

REPO_HF_MODEL = f"{HF_NAME}/{model_name_only}"


In [3]:
import os
import importlib.util

!pip install --upgrade -qqq uv

# Colab thuong da co torch, nhung Qwen3.5 can stack torch 2.8 + Triton tuong ung.
# Cai lai co chu dich tren Colab de tranh torch cu tu runtime truoc.
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"

    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} \
        torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

# Pin theo notebook Qwen3.5 cua Unsloth; --no-deps tranh resolver cai lai torch.
!uv pip install -qqq --upgrade --no-deps \
    "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install -qqq transformers==5.2.0
!uv pip install -qqq --no-build-isolation \
    flash-linear-attention causal_conv1d==1.6.0
!uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"

# T4 (compute capability 7.5) khong can tilelang; A100/L4 co the dung kernel nay.
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install -qqq --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 108.0 MB/s eta 0:00:00


In [4]:
import unsloth

import torch
import transformers

print("Unsloth:", unsloth.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)

/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:86: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: /usr/local/lib/python3.13/dist-packages/torchaudio/lib/_torchaudio.abi3.so: undefined symbol: torch_library_impl
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: 2026.9.4
Torch: 2.8.0+cu128
Transformers: 5.2.0
CUDA: 12.8


In [5]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [9]:
from huggingface_hub import snapshot_download
import json
from pathlib import Path

print(f"📥 Đang tải dữ liệu từ Hugging Face cho [{EXPERIMENT.upper()}] và [benchmark_core]...")
snapshot_download(
    repo_id="ThinhDao/tool-calling-vi-experiments",
    repo_type="dataset",
    local_dir=str(LOCAL_DATA_ROOT),
    allow_patterns=[f"{EXPERIMENT}/**", "benchmark_core/**"],
    ignore_patterns=["**/tool_schema/**"],
    token=True,
)

# ==============================================================================
# 🔍 KIỂM TRA VÀ IN LOG XÁC NHẬN DỮ LIỆU ĐÃ TẢI (DATA VERIFICATION)
# ==============================================================================
exp_dir = LOCAL_DATA_ROOT / EXPERIMENT
train_path = exp_dir / "instruction/train_chat.jsonl"
manifest_path = exp_dir / "manifest.json"

print("\n" + "="*75)
print(f"📂 XÁC NHẬN DỮ LIỆU CHO EXPERIMENT: [{EXPERIMENT.upper()}]")
print("="*75)
print(f"• Thư mục dữ liệu       : {exp_dir}")

if manifest_path.exists():
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        print(f"• File manifest         : {manifest_path.name} (Hợp lệ ✅)")
        if "benchmark_revision" in manifest:
            print(f"• Benchmark Revision    : {manifest['benchmark_revision']}")
        if "description" in manifest:
            print(f"• Mô tả                 : {manifest['description']}")
        if "language" in manifest:
            print(f"• Ngôn ngữ kịch bản     : {manifest['language']}")
    except Exception as e:
        print(f"• File manifest         : Đọc bị lỗi ({e})")
else:
    print(f"• File manifest         : ⚠️ Không tìm thấy manifest.json!")

if train_path.exists():
    total_samples = sum(1 for line in open(train_path, encoding="utf-8") if line.strip())
    size_mb = train_path.stat().st_size / (1024 * 1024)
    print(f"• File Training chính   : {train_path.name}")
    print(f"  └─ Đường dẫn đầy đủ   : {train_path}")
    print(f"  └─ Tổng số mẫu        : {total_samples:,} mẫu")
    print(f"  └─ Dung lượng file    : {size_mb:.2f} MB")

    # In thử 1 câu query thực tế để người dùng đối chiếu ngay ngôn ngữ
    with train_path.open(encoding="utf-8") as f:
        sample_0 = json.loads(f.readline())
        user_query = next((m["content"] for m in sample_0.get("messages", []) if m["role"] == "user"), "N/A")
        tool_count = len(sample_0.get("tools", []))
        fc_count = sum(1 for m in sample_0.get("messages", []) if m.get("tool_calls"))
        print(f"  └─ Kiểm tra mẫu đầu   : Query = '{user_query[:75]}...'")
        print(f"  └─ Cấu trúc mẫu       : {tool_count} tools khả dụng | {fc_count} tool_calls")
    print(f"\n✅ XÁC NHẬN: Dữ liệu nạp vào ĐÚNG CHUẨN cho [{EXPERIMENT.upper()}]!")
else:
    print(f"❌ CẢNH BÁO: Chưa tìm thấy file train tại: {train_path}")
print("="*75 + "\n")

📥 Đang tải dữ liệu từ Hugging Face cho [E4] và [benchmark_core]...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4435 files:   0%|          | 0/4435 [00:00<?, ?it/s]

HTTP Error 429 thrown while requesting GET https://huggingface.co/api/resolve-cache/datasets/ThinhDao/tool-calling-vi-experiments/27562b0100033f433fa645c3fe90b130d9798965/benchmark_core%2F2026-09-02-full-dedup-seed42%2Ftool_schema%2Fretrieve_a_contract.json
Rate limited. Waiting 148.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/ThinhDao/tool-calling-vi-experiments/resolve/27562b0100033f433fa645c3fe90b130d9798965/benchmark_core/2026-09-02-full-dedup-seed42/tool_schema/retrieve_active_loans_offers.json
Rate limited. Waiting 148.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/ThinhDao/tool-calling-vi-experiments/resolve/27562b0100033f433fa645c3fe90b130d9798965/benchmark_core/2026-09-02-full-dedup-seed42/tool_schema/retrieve_buy_sell_informations_by_id.json
Rate limited. Waiting 148.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.c


📂 XÁC NHẬN DỮ LIỆU CHO EXPERIMENT: [E4]
• Thư mục dữ liệu       : /content/data/e4
• File manifest         : manifest.json (Hợp lệ ✅)
• Benchmark Revision    : 2026-09-02-full-dedup-seed42
• File Training chính   : train_chat.jsonl
  └─ Đường dẫn đầy đủ   : /content/data/e4/instruction/train_chat.jsonl
  └─ Tổng số mẫu        : 65,600 mẫu
  └─ Dung lượng file    : 113.38 MB
  └─ Kiểm tra mẫu đầu   : Query = 'Is the email 'john.doe@example.com' part of any known data breaches?...'
  └─ Cấu trúc mẫu       : 2 tools khả dụng | 1 tool_calls

✅ XÁC NHẬN: Dữ liệu nạp vào ĐÚNG CHUẨN cho [E4]!



In [10]:
import unsloth

import argparse
import json
from pathlib import Path
from typing import Any, Iterator

import torch
from datasets import Dataset
from transformers import Trainer, TrainerCallback, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from unsloth import FastLanguageModel


def token_ids(tokenizer: Any, text: str) -> list[int]:
    encoded = tokenizer(text, add_special_tokens=False)
    values = encoded["input_ids"] if isinstance(encoded, dict) else encoded.input_ids
    return list(values[0] if values and isinstance(values[0], list) else values)


def tokenize_assistant_only(
    tokenizer: Any,
    row: dict[str, Any],
    max_seq_length: int,
) -> dict[str, list[int]]:
    messages = row["messages"]
    prompt = tokenizer.apply_chat_template(
        messages[:-1],
        tools=row["tools"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    full = tokenizer.apply_chat_template(
        messages,
        tools=row["tools"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    prompt_ids = token_ids(tokenizer, prompt)
    input_ids = token_ids(tokenizer, full)
    if input_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(f"Prompt is not a prefix for {row.get('id')}")
    if len(input_ids) > max_seq_length:
        input_ids = input_ids[:max_seq_length]
    if len(input_ids) <= len(prompt_ids):
        raise ValueError(f"Assistant target was truncated for {row.get('id')}")
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + input_ids[len(prompt_ids):],
    }


class AssistantOnlyCollator:
    def __init__(self, tokenizer: Any):
        self.tokenizer = tokenizer

    def __call__(self, features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
        max_length = max(len(item["input_ids"]) for item in features)
        pad_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
        return {
            key: torch.tensor(
                [item[key] + [pad_value] * (max_length - len(item[key])) for item in features],
                dtype=torch.long,
            )
            for key, pad_value in (
                ("input_ids", pad_id),
                ("attention_mask", 0),
                ("labels", -100),
            )
        }


class StopAtStep(TrainerCallback):
    def __init__(self, stop_after_step: int | None):
        self.stop_after_step = stop_after_step

    def on_step_end(self, args: Any, state: Any, control: Any, **kwargs: Any) -> Any:
        if self.stop_after_step is not None and state.global_step >= self.stop_after_step:
            control.should_training_stop = True
        return control


def make_training_arguments(kwargs: dict[str, Any]) -> TrainingArguments:
    try:
        return TrainingArguments(**kwargs)
    except TypeError:
        kwargs["group_by_length"] = kwargs.pop("train_sampling_strategy") == "group_by_length"
        return TrainingArguments(**kwargs)


def tokenized_dataset(
    tokenizer: Any,
    path: Path,
    max_seq_length: int,
    limit: int | None,
    num_proc: int = 2,
) -> Dataset:
    raw_dataset = Dataset.from_text(str(path))
    if limit is not None:
        raw_dataset = raw_dataset.select(range(min(limit, len(raw_dataset))))

    def process_row(example: dict[str, Any]) -> dict[str, list[int]]:
        row = json.loads(example["text"])
        return tokenize_assistant_only(tokenizer, row, max_seq_length)

    return raw_dataset.map(
        process_row,
        remove_columns=["text"],
        num_proc=num_proc,
        desc="Tokenizing dataset",
    )


def main(args_list: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-root", required=True)
    parser.add_argument("--experiment", required=True)
    parser.add_argument("--model-id", default="unsloth/Qwen3.5-2B")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--train-limit", type=int, default=None)
    parser.add_argument("--max-seq-length", type=int, default=4096)
    parser.add_argument("--per-device-batch-size", type=int, default=2)
    parser.add_argument("--gradient-accumulation-steps", type=int, default=8)
    parser.add_argument("--stop-after-step", type=int, default=None)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--num-proc", type=int, default=2)
    parser.add_argument("--push-to-hub-repo-id", type=str, default=None)
    args = parser.parse_args(args_list)

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required")
    if args.per_device_batch_size < 1 or args.gradient_accumulation_steps < 1:
        raise ValueError("Batch size and gradient accumulation must be >= 1")

    output_dir = Path(args.run_dir) / "checkpoint"
    train_path = Path(args.data_root) / args.experiment / "instruction/train_chat.jsonl"
    print(f"\n" + "="*70)
    print(f"🚀 [BẮT ĐẦU HUẤN LUYỆN] ĐANG NẠP DỮ LIỆU:")
    print(f"   • Thí nghiệm (Experiment) : {args.experiment.upper()}")
    print(f"   • Model ID                : {args.model_id}")
    print(f"   • File Huấn Luyện         : {train_path}")
    print(f"   • Thư mục Checkpoint      : {output_dir}")
    print("="*70 + "\n")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=args.model_id,
        max_seq_length=args.max_seq_length,
        load_in_4bit=True,
        load_in_16bit=False,
        full_finetuning=False,
    )
    tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=args.seed,
    )
    model.config.use_cache = False
    train_dataset = tokenized_dataset(
        tokenizer,
        train_path,
        args.max_seq_length,
        args.train_limit,
        num_proc=args.num_proc,
    )
    training_kwargs = {
        "output_dir": str(output_dir),
        "per_device_train_batch_size": args.per_device_batch_size,
        "gradient_accumulation_steps": args.gradient_accumulation_steps,
        "num_train_epochs": 1.0,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "logging_first_step": True,
        "disable_tqdm": False,
        "save_strategy": "steps",
        "save_steps": 250,
        "save_total_limit": 2,
        "report_to": "none",
        "remove_unused_columns": False,
        "gradient_checkpointing": True,
        "optim": "adamw_8bit",
        "seed": args.seed,
        "fp16": not torch.cuda.is_bf16_supported(),
        "bf16": torch.cuda.is_bf16_supported(),
        "dataloader_num_workers": 0,
        "dataloader_pin_memory": False,
        "train_sampling_strategy": "group_by_length",
        "skip_memory_metrics": True,
    }
    trainer = Trainer(
        model=model,
        args=make_training_arguments(training_kwargs),
        train_dataset=train_dataset,
        data_collator=AssistantOnlyCollator(tokenizer),
        callbacks=[StopAtStep(args.stop_after_step)],
    )
    checkpoint = get_last_checkpoint(str(output_dir))
    effective_batch_size = (
        args.per_device_batch_size * args.gradient_accumulation_steps * 1
    )
    print(
        f"world_size=1 per_device_batch_size={args.per_device_batch_size} "
        f"gradient_accumulation_steps={args.gradient_accumulation_steps} "
        f"effective_batch_size={effective_batch_size} resume_from={checkpoint} "
        f"stop_after_step={args.stop_after_step}"
    )
    trainer.train(resume_from_checkpoint=checkpoint)
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print("training stage complete:", output_dir)

    # New: Push to Hugging Face Hub using direct unsloth capability
    if args.push_to_hub_repo_id:
        print(f"\n🚀 Pushing model to Hugging Face Hub: {args.push_to_hub_repo_id}")
        model.push_to_hub(args.push_to_hub_repo_id)
        tokenizer.push_to_hub(args.push_to_hub_repo_id)
        print("✅ Model successfully pushed to Hugging Face Hub.")

In [11]:
!nproc


12


In [ ]:
import multiprocessing

PER_DEVICE_BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 1

TRAIN_LIMIT = None
STOP_AFTER_STEP = None

# Tự động lấy số core CPU (ví dụ: sẽ là 12 trên A100)
# Thường trừ đi 1 hoặc 2 core để tránh bị nghẽn hệ thống (ví dụ: dùng 10 hoặc 11 core)
NUM_PROC = max(2, multiprocessing.cpu_count() - 1)

train_args = [
    "--experiment", EXPERIMENT,
    "--model-id", MODEL_ID,
    "--run-dir", str(LOCAL_RUN_DIR),
    "--data-root", str(LOCAL_DATA_ROOT),
    "--per-device-batch-size", str(PER_DEVICE_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--num-proc", str(NUM_PROC),
    "--push-to-hub-repo-id", str(REPO_HF_MODEL)
]

if TRAIN_LIMIT is not None:
    train_args.extend(["--train-limit", str(TRAIN_LIMIT)])

if STOP_AFTER_STEP is not None:
    train_args.extend(["--stop-after-step", str(STOP_AFTER_STEP)])

main(train_args)


🚀 [BẮT ĐẦU HUẤN LUYỆN] ĐANG NẠP DỮ LIỆU:
   • Thí nghiệm (Experiment) : E4
   • Model ID                : unsloth/Qwen3.5-2B
   • File Huấn Luyện         : /content/data/e4/instruction/train_chat.jsonl
   • Thư mục Checkpoint      : /content/run/checkpoint

==((====))==  Unsloth 2026.9.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Generating train split: 0 examples [00:00, ? examples/s]

Tokenizing dataset (num_proc=11):   0%|          | 0/65600 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


world_size=1 per_device_batch_size=64 gradient_accumulation_steps=1 effective_batch_size=64 resume_from=None stop_after_step=None


In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

# Đường dẫn checkpoint cục bộ
LOCAL_CHECKPOINT_DIR = "/content/run/checkpoint"

print(f"🔄 Đang tải adapter từ: {LOCAL_CHECKPOINT_DIR}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LOCAL_CHECKPOINT_DIR,
    max_seq_length=4096,
    load_in_4bit=True,
)

print(f"🚀 Bắt đầu đẩy adapter lên Hugging Face Hub: {REPO_HF_MODEL}")
model.push_to_hub(REPO_HF_MODEL)
tokenizer.push_to_hub(REPO_HF_MODEL)
print("✅ Đã đẩy adapter và tokenizer lên Hugging Face thành công!")

In [ ]:
from huggingface_hub import HfApi
try:
    user_info = HfApi().whoami()
    print(f"🔑 Token hiện tại thuộc về tài khoản: {user_info['name']}")
    print(f"📌 Email: {user_info.get('email', 'N/A')}")
    print(f"📌 Quyền hạn Token (Auth type): {user_info.get('auth', {}).get('type', 'N/A')}")
except Exception as e:
    print(f"❌ Không thể xác thực Token. Lỗi: {e}")